# 05 — The curve Jacobian: zero risk → par risk by the implicit-function theorem

## The idea

A price $V$ depends on the curve nodes $z$; the trader hedges with the quoted instruments $q$. The bootstrap
defines $z$ implicitly through $F(z(q);\,q) = 0$. Differentiate that identity:

$$\frac{\partial F}{\partial z}\,\frac{dz}{dq} + \frac{\partial F}{\partial q} = 0
\quad\Rightarrow\quad
\frac{dz}{dq} = -\left(\frac{\partial F}{\partial z}\right)^{-1}\frac{\partial F}{\partial q},$$

and the par risk is $\dfrac{\partial V}{\partial q} = \dfrac{\partial V}{\partial z}\,\dfrac{dz}{dq}$.
Computationally: one transposed solve, $\left(\partial F/\partial z\right)^{\!T} w = \left(\partial V/\partial z\right)^{\!T}$,
then $\partial V/\partial q = -w^{T}\,\partial F/\partial q$. Never form an inverse; never re-bootstrap $N$ times.

Sensitivities (adjoint, or bumps here) stop at $z$; the Jacobian carries them to $q$. It is exact to the
extent the bootstrap is converged — the theorem differentiates the *equation*, not the iteration.

## A 2×2 hand example

Take the two-node curve of notebook 04 in rate form: $F_1 = K_1^{par}(z_1) - q_1$, $F_2 = K_2^{par}(z_1,z_2) - q_2$
with $z_i = -\log P_i$. Then $\partial F/\partial q = -I$, $\partial F/\partial z$ is lower-triangular (the 1Y does
not see the 2Y node), and $dz/dq = (\partial F/\partial z)^{-1}$. Compute $\partial F/\partial z$ by finite
differences in Python, then check $dz/dq$ against a re-bootstrap with $q_2$ bumped by one basis point.

In [ ]:
import _util
from _util import run_demo
import numpy as np, matplotlib.pyplot as plt

tau1, tau2 = 365/360, 366/360
def bootstrap(q1, q2):
    P1 = 1 / (1 + q1 * tau1)
    P2 = (1 - q2 * tau1 * P1) / (1 + q2 * tau2)
    return np.array([-np.log(P1), -np.log(P2)])
def par_rates(z):
    P1, P2 = np.exp(-z)
    return np.array([(1 / P1 - 1) / tau1, (1 - P2) / (tau1 * P1 + tau2 * P2)])
q = np.array([0.0348, 0.0332]); z = bootstrap(*q)
h = 1e-7; J = np.zeros((2, 2))
for j in range(2):
    e = np.zeros(2); e[j] = h
    J[:, j] = (par_rates(z + e) - par_rates(z - e)) / (2 * h)
dzdq_ift = np.linalg.solve(J, np.eye(2))          # -J^{-1} (dF/dq) with dF/dq = -I
dzdq_bump = np.column_stack([(bootstrap(q[0] + 1e-4, q[1]) - z) / 1e-4, (bootstrap(q[0], q[1] + 1e-4) - z) / 1e-4])
print("dF/dz =\n", J.round(4)); print("dz/dq (IFT) =\n", dzdq_ift.round(4)); print("dz/dq (bump 1bp) =\n", dzdq_bump.round(4))
print("max |diff| =", np.abs(dzdq_ift - dzdq_bump).max())

## Par risk of a 10y payer swap on the full two-curve set

`rl_demo_bootstrap parrisk` prices a 10y SOFR payer at 3.00% (1e6 notional), takes $\partial V/\partial z$ by
central differences, maps it through the bootstrap Jacobian, and compares with
bumping each quote by a basis point and re-bootstrapping.

In [ ]:
pr = run_demo("rl_demo_bootstrap", "parrisk", "linear", 10, 0.03)
print("max |IFT - bump| per bp:", pr["diff"].abs().max())
fig, ax = plt.subplots(figsize=(11, 4))
x = np.arange(len(pr))
ax.bar(x - 0.2, pr["par_risk_ift"], 0.4, label="IFT (analytic Jacobian)")
ax.bar(x + 0.2, pr["par_risk_bump"], 0.4, label="bump & rebuild")
ax.set_xticks(x); ax.set_xticklabels(pr["id"], rotation=90, fontsize=7); ax.set_ylabel("PV change per bp of quote")
ax.legend(); ax.set_title("Par risk of a 10y SOFR payer: the 10y OIS quote carries it")
plt.tight_layout(); plt.show()
pr.sort_values("par_risk_ift", key=abs, ascending=False).head(8)

## The sparsity of ∂F/∂z is the picture of the coupling

Rows are instruments, columns are nodes (SOFR block then FF block). SOFR-only rows are zero in the FF block;
the FF OIS and basis-swap rows touch both. The two basis-swap rows assigned to the SOFR curve are the reason
notebook 04's sequential pass fails.

In [ ]:
jac = run_demo("rl_demo_bootstrap", "jacobian", "linear")
rows = list(dict.fromkeys(jac["row_id"]))
M = jac.pivot(index="row_id", columns="col", values="value").reindex(rows)
fig, ax = plt.subplots(figsize=(8, 7))
im = ax.imshow(np.log10(np.abs(M.values) + 1e-16), cmap="viridis", aspect="auto")
ax.set_yticks(range(len(rows))); ax.set_yticklabels(rows, fontsize=6); ax.set_xlabel("node (SOFR block, then FF block)")
plt.colorbar(im, label="log10 |dF_i/dz_j|"); ax.set_title("Bootstrap Jacobian sparsity"); plt.tight_layout(); plt.show()